In [ ]:
# Option 2: Batch test
batch_size = 5
raw_dir = project_root / "data" / "raw" / "images"
clahe_dir = project_root / "data" / "clahe_cache"
image_list = list(raw_dir.glob("*.png")) if raw_dir.exists() else []
if not image_list:
    image_list = list(clahe_dir.glob("*.png")) if clahe_dir.exists() else []
if not image_list:
    raise FileNotFoundError("No images found in data/raw/images or data/clahe_cache")

for img_path in image_list[:batch_size]:
    top1 = predict_image(img_path, top_k=1)[0]
    print(f"{img_path.name:<40} {top1[0]:<25} {top1[1]:.4f}")

In [ ]:
# Option 1: Single image
# Set this to a file you want, or leave None to auto-pick
image_path = None

if image_path is None:
    raw_dir = project_root / "data" / "raw" / "images"
    clahe_dir = project_root / "data" / "clahe_cache"
    candidates = list(raw_dir.glob("*.png")) if raw_dir.exists() else []
    if not candidates:
        candidates = list(clahe_dir.glob("*.png")) if clahe_dir.exists() else []
    if not candidates:
        raise FileNotFoundError("No images found in data/raw/images or data/clahe_cache")
    image_path = candidates[0]

print(f"Testing single image: {image_path.name}")
top5 = predict_image(image_path, top_k=5)
for label, prob in top5:
    print(f"{label:<25} {prob:.4f}")

In [7]:
import sys
from pathlib import Path
import torch
import numpy as np
from PIL import Image

# Resolve project root
def find_project_root(start: Path) -> Path:
    markers = {"setup.py", "requirements.txt", "README.md"}
    for path in [start] + list(start.parents):
        if any((path / m).exists() for m in markers):
            return path
    return start

project_root = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(project_root))

from config import DISEASE_LABELS
from ml.models.student_model import create_student_model, MODEL_CONFIGS
from ml.data.preprocessing import get_medical_transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Find checkpoint in ml/models/checkpoints/final
final_dir = project_root / "ml" / "models" / "checkpoints" / "final"
ckpts = list(final_dir.glob("*.pth")) if final_dir.exists() else []
if not ckpts:
    raise FileNotFoundError(f"No .pth found in {final_dir}")
checkpoint_path = ckpts[0]
print(f"Using checkpoint: {checkpoint_path.name}")

# Load checkpoint and model
ckpt = torch.load(checkpoint_path, map_location='cpu')
model_arch = ckpt.get('model_arch')
if not model_arch and checkpoint_path.parent.name in MODEL_CONFIGS:
    model_arch = checkpoint_path.parent.name
if not model_arch:
    model_arch = "convnext_tiny_mhsa"
model = create_student_model(model_arch, num_classes=14, pretrained=False)
model.load_state_dict(ckpt['model_state_dict'])
model.to(device).eval()

transform = get_medical_transforms(use_clahe=True, use_denoising=False)

def predict_image(image_path: Path, top_k: int = 5):
    image = Image.open(image_path).convert('RGB')
    x = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.sigmoid(model(x)).squeeze(0).cpu().numpy()
    idx = np.argsort(probs)[::-1][:top_k]
    return [(DISEASE_LABELS[i], float(probs[i])) for i in idx]

Device: cuda


FileNotFoundError: No .pth found in C:\Users\User\Sadeepa\X-Lite\notebooks\ml\models\checkpoints\final

# X-Lite Quick Inference (No FE/BE)
Two options only: single image and batch.